<a href="https://colab.research.google.com/github/Kamalll7/Final_Project_PCD-Kelompok_6/blob/main/notebook/02_Number_Plate_Detection_ROI_HOG_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Pendahuluan

### 1.1 Latar Belakang Singkat

Undang-Undang Nomor 22 Tahun 2009 mewajibkan kendaraan bermotor yang beroperasi di jalan
dilengkapi **Tanda Nomor Kendaraan Bermotor (TNKB)**. Deteksi keberadaan plat nomor merupakan
langkah awal yang krusial dalam sistem *Automatic License Plate Recognition* (ALPR), sebelum
tahap segmentasi dan pengenalan karakter.

### 1.2 Rumusan Masalah yang Dijawab Notebook Ini

> **Bagaimana mendeteksi keberadaan plat nomor pada sepeda motor menggunakan metode
> pengolahan citra digital dan machine learning?** (Rumusan Masalah #2)

### 1.3 Tujuan

Mengembangkan dan mengevaluasi metode deteksi keberadaan plat nomor sepeda motor menggunakan
**RoI** untuk mempersempit area pencarian, **HOG** untuk mengekstraksi deskriptor tepi/gradien,
dan **SVM** untuk klasifikasi biner.

### 1.4 Mengapa Deteksi Plat Motor Lebih Sulit daripada Plat Mobil?

Proposal §2.6 mencatat beberapa tantangan spesifik:

1. **Latar belakang kompleks** — plat motor bercampur dengan bodi kendaraan, knalpot, rantai,
   dan elemen lain di sekitarnya, berbeda dengan plat mobil yang relatif bersih.
2. **Ukuran kecil dan orientasi bervariasi** — plat motor lebih kecil dan dapat miring
   (*yaw*, *pitch*, *roll*).
3. **Literatur lebih jarang** — deteksi plat motor jauh lebih sedikit diteliti dibanding plat mobil.

### 1.5 Strategi: RoI Dulu, Baru Cari Plat

Hsu et al. (2015) — dirujuk proposal §2.6 — membandingkan dua skema:

| Skema | Cara kerja | Hasil |
|---|---|---|
| **Deteksi motor dulu**, lalu cari plat di dalamnya | meniru cara kerja visual manusia | **false positive jauh lebih sedikit**, deteksi lebih cepat |
| Deteksi plat langsung pada seluruh citra | seperti *sliding window* pada piramida citra | lebih banyak false positive, lebih lambat |

Contoh angka dari studi tersebut: deteksi motor **279 ms**, lalu plat hanya butuh **~8 ms**
tambahan di dalam area motor — jauh lebih cepat dibanding memindai seluruh citra.

**Karena itu notebook ini mengadopsi skema pertama**: RoI ditentukan lebih dulu (dari anotasi
`rider`), lalu pencarian plat dilakukan **hanya di dalam RoI tersebut**.

---
## 2. Setup Environment

Dirancang untuk **Google Colab** (sesuai §3.10 proposal) maupun eksekusi lokal.

**Perangkat lunak** (§3.10): `Python`, `OpenCV`, `scikit-image` (HOG), `scikit-learn` (SVM & metrik),
`NumPy`, `Matplotlib`, `Pandas`.

In [ ]:
import sys, subprocess

REQUIRED = ["opencv-python-headless", "scikit-image", "scikit-learn",
            "matplotlib", "pandas", "numpy", "pillow"]

def _ensure(pkgs):
    missing = []
    for p in pkgs:
        mod = {"opencv-python-headless": "cv2", "scikit-image": "skimage",
               "scikit-learn": "sklearn", "pillow": "PIL"}.get(p, p)
        try:
            __import__(mod)
        except ImportError:
            missing.append(p)
    if missing:
        print("Memasang:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_ensure(REQUIRED)
print("Dependensi siap.")

In [ ]:
import os, json, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score,
                             roc_curve, auc, classification_report,
                             precision_recall_curve, average_precision_score)

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

import sklearn, skimage
print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)

---
## 3. Akuisisi Dataset

Dataset yang sama dengan Jalur 1, dengan struktur anotasi **YOLO**:

```
dataset/
├── train/{images,labels}/
└── val/{images,labels}/
```

**Kelas:**

| ID | Nama kelas |
|---|---|
| 0 | `with helmet` |
| 1 | `without helmet` |
| 2 | `rider` |
| 3 | `number plate` |

Kelas yang dipakai notebook ini: **`rider`** (untuk menentukan RoI) dan
**`number plate`** (sebagai *ground truth*).

In [ ]:
DATASET_SLUG = "aneesarom/rider-with-helmet-without-helmet-number-plate"
CLASS_NAMES  = ["with helmet", "without helmet", "rider", "number plate"]

def _looks_like_dataset(p: Path) -> bool:
    return (p / "train" / "images").is_dir() and (p / "train" / "labels").is_dir()

def resolve_dataset() -> Path:
    candidates = []
    env = os.environ.get("PCD_DATASET_DIR")
    if env:
        candidates.append(Path(env))
    candidates += [Path("./dataset"), Path("../dataset"), Path("/content/dataset"),
                   Path.home() / ".cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate"]
    for base in candidates:
        if not base.exists():
            continue
        if _looks_like_dataset(base):
            return base
        for sub in base.rglob("train"):
            if (sub / "images").is_dir() and (sub / "labels").is_dir():
                return sub.parent
    print("Dataset belum ditemukan secara lokal, mengunduh via kagglehub ...")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    for sub in [root] + list(root.rglob("*")):
        if _looks_like_dataset(sub):
            return sub
    raise FileNotFoundError("Struktur dataset tidak dikenali setelah unduhan.")

DATA_ROOT = resolve_dataset()
print("Dataset root :", DATA_ROOT.resolve())
print("Train images :", len(list((DATA_ROOT / 'train' / 'images').glob('*'))))
print("Val images   :", len(list((DATA_ROOT / 'val' / 'images').glob('*'))))

### 3.1 Pembaca Citra yang Robust

Satu berkas berekstensi `.jpg` pada dataset sebenarnya berformat **AVIF**, sehingga `cv2.imread()`
mengembalikan `None`. Digunakan pembaca dengan *fallback* ke `Pillow` agar tidak ada data yang
hilang secara diam-diam.

In [ ]:
def imread_robust(path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None:
        return img
    from PIL import Image
    with Image.open(path) as im:
        arr = np.array(im.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def parse_label_file(path: Path):
    boxes = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        boxes.append({"cls": cls,
                      "x1": x - w / 2, "y1": y - h / 2,
                      "x2": x + w / 2, "y2": y + h / 2})
    return boxes


def build_index(root: Path) -> pd.DataFrame:
    rows = []
    for split in ("train", "val"):
        for lbl in sorted((root / split / "labels").glob("*.txt")):
            stem = lbl.stem
            img_path = root / split / "images" / (stem + ".jpg")
            if not img_path.exists():
                hits = list((root / split / "images").glob(stem + ".*"))
                if not hits:
                    continue
                img_path = hits[0]
            boxes = parse_label_file(lbl)
            row = {"split": split, "stem": stem, "image_path": str(img_path),
                   "n_objects": len(boxes), "boxes": boxes}
            for cid, cname in enumerate(CLASS_NAMES):
                row["n_" + cname] = sum(1 for b in boxes if b["cls"] == cid)
            rows.append(row)
    return pd.DataFrame(rows)

df = build_index(DATA_ROOT)
print(f"Total citra terindeks: {len(df)}")

bad = [p for p in df["image_path"] if imread_robust(p) is None]
print("Citra gagal dibaca:", len(bad))

total = {c: int(df["n_" + c].sum()) for c in CLASS_NAMES}
print("\nJumlah bounding box per kelas:")
for k, v in total.items():
    print(f"  {k:16s} : {v}")